In [1]:
import os
import tempfile
from collections import defaultdict

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from osgeo import gdal, ogr
from shapely.strtree import STRtree

import mnk.substrat as subkart
import mnk.utils
import mnk.vectorize

gdal.UseExceptions()


## Final Substrat dataproduct: merge authoritative classifications

Replace model predictions with authoritative polygons from the *klassifisering* dataset.
For klass, `blanding` (DN=1) is remapped to `løsbunn` (DN=0) since the model has no
blanding class, but `LM_DK` retains `DK_0` for those polygons. Predictions arrive from
04_postprocess already carrying `BunnType`, `LM_DK`, and `Sannsynlighet_LM_DK` (blanding pixels
are resolved to hard/soft per polygon by highest probability, with `LM_DK == DK_0`).
Probability is set to 100 for all authoritative polygons.

In [2]:
nodata = 255
crs = "EPSG:25833"
classifier = subkart.utils.load_classifier()

In [3]:
fname = mnk.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "norge", "latest", crs.split(":")[1]
)

In [4]:
# LM_DK habitat codes per BunnType (shared source of truth).
# Assigned to klass *before* the blanding→løsbunn remap so those polygons keep DK_0.
LM_DK_MAP = subkart.labelling.LM_DK_MAP

# Load authoritative classifications – only needed columns
klass = gpd.read_parquet(
    "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-klassifisering_norge_latest_25833.geo.parquet",
    columns=["BunnType", "geometry"],
)

# Assign LM_DK before remapping blanding so blanding keeps DK_0
klass["LM_DK"] = klass["BunnType"].map(LM_DK_MAP)

# Map blanding (not in model) → løsbunn
klass["BunnType"] = klass["BunnType"].replace({"blanding": "løsbunn"})
klass["DN"] = klass["BunnType"].map({"løsbunn": 0, "fastbunn": 2})
klass["Sannsynlighet_LM_DK"] = 100.0
klass["Kilde"] = "NGU - Bunnsedimenter (kornstørrelse), detaljert"

# Load model predictions (padded version for clean land subtraction).
# LM_DK is already populated in 04_postprocess (blanding polygons carry DK_0).
pred = gpd.read_parquet(
    f"gs://niva-geodata/MarintNaturKart/results/{fname}_padded.geo.parquet",
    columns=["DN", "BunnType", "LM_DK", "Sannsynlighet_LM_DK", "geometry"],
)
pred["Kilde"] = "NIVA - Substrat Modell"



In [5]:
# --- Fast erase: bulk spatial join finds all intersecting pairs in one GEOS call ---
klass_geoms = klass.geometry.values
pred_geoms = pred.geometry.values.copy()

tree = STRtree(klass_geoms)
pred_hit_idxs, klass_hit_idxs = tree.query(pred_geoms, predicate="intersects")

# Group klass indices by pred polygon for efficient per-polygon difference
pred_to_klass = defaultdict(list)
for p, k in zip(pred_hit_idxs.tolist(), klass_hit_idxs.tolist()):
    pred_to_klass[p].append(k)

n_clip = len(pred_to_klass)
print(f"Clipping {n_clip:,} of {len(pred_geoms):,} prediction polygons...")
for i, (pred_i, klass_is) in enumerate(pred_to_klass.items(), 1):
    if i % 1000 == 0:
        print(f"  {i:,} / {n_clip:,}")
    eraser = shapely.unary_union(klass_geoms[klass_is])
    pred_geoms[pred_i] = shapely.difference(pred_geoms[pred_i], eraser)

pred = pred.set_geometry(pred_geoms)
pred_remaining = pred[~pred.geometry.is_empty].explode(index_parts=False)

COLS = ["DN", "BunnType", "LM_DK", "Sannsynlighet_LM_DK", "Kilde", "geometry"]
final = pd.concat(
    [pred_remaining[COLS], klass[COLS]],
    ignore_index=True,
)



Clipping 45,896 of 328,517 prediction polygons...
  1,000 / 45,896
  2,000 / 45,896
  3,000 / 45,896
  4,000 / 45,896
  5,000 / 45,896
  6,000 / 45,896
  7,000 / 45,896
  8,000 / 45,896
  9,000 / 45,896
  10,000 / 45,896
  11,000 / 45,896
  12,000 / 45,896
  13,000 / 45,896
  14,000 / 45,896
  15,000 / 45,896
  16,000 / 45,896
  17,000 / 45,896
  18,000 / 45,896
  19,000 / 45,896
  20,000 / 45,896
  21,000 / 45,896
  22,000 / 45,896
  23,000 / 45,896
  24,000 / 45,896
  25,000 / 45,896
  26,000 / 45,896
  27,000 / 45,896
  28,000 / 45,896
  29,000 / 45,896
  30,000 / 45,896
  31,000 / 45,896
  32,000 / 45,896
  33,000 / 45,896
  34,000 / 45,896
  35,000 / 45,896
  36,000 / 45,896
  37,000 / 45,896
  38,000 / 45,896
  39,000 / 45,896
  40,000 / 45,896
  41,000 / 45,896
  42,000 / 45,896
  43,000 / 45,896
  44,000 / 45,896
  45,000 / 45,896


In [6]:
fname_final = mnk.utils.to_filename("nisjedata-substrat", "norge", "2026", crs.split(":")[1])

## Clip to grunnlinje (1 nautisk mil)

Remove polygons outside the baseline + 1 nautical mile boundary.

In [7]:
print(f"Before grunnlinje clip: {len(final):,} polygons")

with tempfile.TemporaryDirectory() as tmpdir:
    tmp_in = os.path.join(tmpdir, "final_unclipped.gpkg")
    tmp_out = os.path.join(tmpdir, "final_clipped.gpkg")
    final.to_file(tmp_in, driver="GPKG", layer="data")

    grunnlinje_url = "/vsicurl/https://storage.googleapis.com/niva-geodata/MarintNaturKart/aux/grunnlinje_1nautisk.gpkg"
    gdal.VectorTranslate(
        tmp_out, tmp_in,
        options=gdal.VectorTranslateOptions(clipSrc=grunnlinje_url),
    )
    final = gpd.read_file(tmp_out).explode(index_parts=False).reset_index(drop=True)

final = final[~final.is_empty].reset_index(drop=True)
print(f"After grunnlinje clip:  {len(final):,} polygons")

# Drop clip artifacts: gdal.VectorTranslate(clipSrc=...) can emit sliver polygons
# with all attributes NULL when the clip line dissects a polygon awkwardly.
attr_cols = ["DN", "BunnType", "LM_DK", "Sannsynlighet_LM_DK", "Kilde"]
null_mask = final[attr_cols].isna().all(axis=1)
print(f"Clip artifacts (all-null rows): {int(null_mask.sum()):,}")
final = final[~null_mask].reset_index(drop=True)

Before grunnlinje clip: 446,067 polygons
After grunnlinje clip:  440,638 polygons
Clip artifacts (all-null rows): 0


## Subtract land

Remove land areas from the final polygons using the same approach
as in `02_light_attenuation.ipynb`.

In [8]:
land = gpd.read_parquet(
    "gs://niva-geodata/MarintNaturKart/aux/Basisdata_Landareal.geo.parquet"
)
final = mnk.vectorize.subtract_land(final, land)
del land
print(f"After land subtraction: {len(final):,} polygons")

Subtracting land from 76,669 of 440,638 polygons...
After land subtraction: 440,310 polygons


## Save final product

In [9]:
final.to_parquet(f"{fname_final}.geo.parquet", compression="snappy")
final.to_file(f"{fname_final}.gpkg", driver="GPKG", layer="bunntyper")
print(f"Saved: {fname_final} ({len(final):,} polygons)")

Saved: nisjedata-substrat_norge_2026_25833 (440,310 polygons)


In [10]:
mnk.utils.to_postgis(final, fname_final)

Table nisjedata_substrat_norge_2026 uploaded to PostGIS.
